In [1]:
# Upgrade transformers to support Gemma 4
!pip install -U transformers accelerate bitsandbytes

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForMultimodalLM, BitsAndBytesConfig, TextStreamer

# The path you confirmed

# for Gemma 4 12b the path on kaggle is :: /kaggle/input/models/google/gemma-4/transformers/gemma-4-12b/2
model_id = "/kaggle/input/models/google/gemma-4/transformers/gemma-4-31b-it/1"

# 1. Quantization to fit 31B on Kaggle's GPU
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

# 2. Load with Multi-Modal support (Standard for Gemma 4)
processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForMultimodalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# 3. Prompt with Thinking Mode
messages = [
    {"role": "user", "content": [{"type": "text", "text": "Solve this ARC task. Explain your reasoning in detail inside a thinking block."}]}
]

prompt = processor.apply_chat_template(
    messages, 
    tokenize=False, 
    add_generation_prompt=True,
    enable_thinking=True
)

inputs = processor(text=prompt, return_tensors="pt").to(model.device)
streamer = TextStreamer(processor)

print("--- Model is thinking... ---")
outputs = model.generate(
    **inputs, 
    streamer=streamer, 
    max_new_tokens=2048, 
    do_sample=True,
    temperature=0.7
)

Loading weights:   0%|          | 0/1188 [00:00<?, ?it/s]